In [17]:
states = ["Code", "Test", "Dependency", "CI/Config"] 

actions = [
    "Inspect pipeline logs",
    "Inspect failed pipeline stage",
    "Compare with last successful run",
    "Inspect recent code/test changes",
    "Inspect dependency changes",
    "Inspect CI/environment configuration",
    "Search previous incidents/runbooks",
    "Inspect Docker image/cache changes",
    "Inspect hosting environment",
    "Inspect server/session state",
    "Escalate to human"
]


priors = {
    "Code": 0.1770,
    "Test": 0.3934,
    "Dependency": 0.2984,
    "CI/Config": 0.1311
}

likelihoods = {
    "Build": {
        "Code": 0.574,
        "Test": 0.358,
        "Dependency": 0.429,
        "CI/Config": 0.275
    },
    "Test": {
        "Code": 0.185,
        "Test": 0.533,
        "Dependency": 0.132,
        "CI/Config": 0.100
    },
    "Setup / Dependency": {
        "Code": 0.074,
        "Test": 0.000,
        "Dependency": 0.176,
        "CI/Config": 0.125
    },
    "Quality / Analysis": {
        "Code": 0.019,
        "Test": 0.058,
        "Dependency": 0.055,
        "CI/Config": 0.050
    },
    "Deploy / Publish": {
        "Code": 0.000,
        "Test": 0.000,
        "Dependency": 0.033,
        "CI/Config": 0.050
    },
    "Ambiguous": {
        "Code": 0.074,
        "Test": 0.033,
        "Dependency": 0.077,
        "CI/Config": 0.150
    },
    "Unknown": {
        "Code": 0.074,
        "Test": 0.017,
        "Dependency": 0.099,
        "CI/Config": 0.250
    }
}

In [18]:
def calculate_posterior(failed_stage):
    stage_likelihoods = likelihoods[failed_stage]

    unnormalized = {}

    for state in states:
        unnormalized[state] = (
            priors[state] * stage_likelihoods[state]
        )

    evidence_probability = sum(unnormalized.values())

    posterior = {}

    for state in states:
        posterior[state] = (
            unnormalized[state] / evidence_probability
        )

    return posterior

In [19]:
posterior = calculate_posterior("Build")

for state, probability in posterior.items():
    print(f"{state}: {probability:.3%}")

Code: 24.993%
Test: 34.646%
Dependency: 31.492%
CI/Config: 8.869%


In [20]:
CONFIDENCE_THRESHOLD = 0.90


def should_stop(posterior):
    best_state = max(posterior, key=posterior.get)
    best_probability = posterior[best_state]

    if best_probability >= CONFIDENCE_THRESHOLD:
        return True, best_state

    return False, best_state

In [21]:
posterior = calculate_posterior("Build")

stop, best_state = should_stop(posterior)

print("Best hypothesis:", best_state)
print("Stop diagnosis:", stop)

Best hypothesis: Test
Stop diagnosis: False


In [22]:
actions = [
    {
        "name": "Inspect pipeline logs",
        "tags": ["all"],
        "cost": 1,
        "outcomes": [
            "Relevant error found",
            "No clear error found"
        ]
    },
    {
        "name": "Inspect failed pipeline stage",
        "tags": ["all"],
        "cost": 2,
        "outcomes": [
            "Specific failed stage identified",
            "Failure remains ambiguous"
        ]
    },
    {
        "name": "Compare with last successful run",
        "tags": ["all"],
        "cost": 2,
        "outcomes": [
            "Significant difference found",
            "No significant difference found"
        ]
    },
    {
        "name": "Inspect recent code/test changes",
        "tags": ["Build", "Test", "Quality / Analysis"],
        "cost": 1,
        "outcomes": [
            "Relevant code/test change found",
            "No relevant change found"
        ]
    },
    {
        "name": "Inspect dependency changes",
        "tags": ["Build", "Setup / Dependency"],
        "cost": 1,
        "outcomes": [
            "Dependency change found",
            "No dependency change found"
        ]
    },
    {
        "name": "Inspect CI/environment configuration",
        "tags": [
            "Build",
            "Test",
            "Setup / Dependency",
            "Quality / Analysis",
            "Deploy / Publish"
        ],
        "cost": 2.5,
        "outcomes": [
            "Configuration/environment issue found",
            "No configuration/environment issue found"
        ]
    },
    {
        "name": "Search previous incidents/runbooks",
        "tags": ["all"],
        "cost": 1,
        "outcomes": [
            "Similar incident found",
            "No similar incident found"
        ]
    },
    {
        "name": "Inspect Docker image/cache changes",
        "tags": ["Build", "Deploy / Publish"],
        "cost": 1.5,
        "outcomes": [
            "Docker/image difference found",
            "No Docker/image difference found"
        ]
    },
    {
        "name": "Inspect hosting environment",
        "tags": ["Deploy / Publish"],
        "cost": 2,
        "outcomes": [
            "Infrastructure issue found",
            "No infrastructure issue found"
        ]
    },
    {
        "name": "Inspect server/session state",
        "tags": ["Build", "Deploy / Publish"],
        "cost": 1.5,
        "outcomes": [
            "Server/session issue found",
            "No server/session issue found"
        ]
    },
    {
        "name": "Escalate to human",
        "tags": ["all"],
        "cost": 2.5,
        "outcomes": [
            "Human investigation initiated"
        ]
    }
]

import math

def entropy(probabilities):
    return -sum(
        p * math.log2(p)
        for p in probabilities
        if p > 0
    )


def calculate_eig(current_posterior, outcome_model):
    current_entropy = entropy(current_posterior.values())

    eig = 0

    for outcome, probabilities in outcome_model.items():

        # P(outcome | E, action)
        outcome_probability = sum(
            current_posterior[state] * probabilities[state]
            for state in states
        )

        # P(H | E, outcome, action)
        outcome_posterior = {}

        for state in states:
            numerator = (
                current_posterior[state]
                * probabilities[state]
            )

            outcome_posterior[state] = (
                numerator / outcome_probability
            )

        # Information gained from this outcome
        outcome_entropy = entropy(outcome_posterior.values())

        information_gain = (
            current_entropy - outcome_entropy
        )

        # Expected contribution of this outcome
        eig += outcome_probability * information_gain

    return eig

def rank_actions(posterior, candidate_actions, outcome_models):
    ranked_actions = []

    for action in candidate_actions:
        name = action["name"]
        cost = action["cost"]

        eig = calculate_eig(
            posterior,
            outcome_models[name]
        )

        score = eig / cost

        ranked_actions.append({
            "name": name,
            "cost": cost,
            "eig": eig,
            "score": score
        })

    ranked_actions.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return ranked_actions


def get_candidate_actions(failed_stage, actions):
    candidate_actions = []

    for action in actions:
        if "all" in action["tags"] or failed_stage in action["tags"]:
            candidate_actions.append(action)

    return candidate_actions

In [23]:
outcome_models = {
    "Inspect recent code/test changes": {
        "Relevant change found": {
            "Code": 0.70,
            "Test": 0.30,
            "Dependency": 0.20,
            "CI/Config": 0.10,
        },
        "No relevant change found": {
            "Code": 0.30,
            "Test": 0.70,
            "Dependency": 0.80,
            "CI/Config": 0.90,
        },
    }
}

candidate_actions = [
    {
        "name": "Inspect recent code/test changes",
        "tags": ["Build", "Test", "Quality / Analysis"],
        "cost": 1,
    }
]

posterior = calculate_posterior("Build")

ranked = rank_actions(
    posterior,
    candidate_actions,
    outcome_models
)

for action in ranked:
    print(
        action["name"],
        action["eig"],
        action["cost"],
        action["score"]
    )

Inspect recent code/test changes 0.14019170448155632 1 0.14019170448155632


In [24]:
failed_stage = "Build"

candidate_actions = get_candidate_actions(
    failed_stage,
    actions
)

for action in candidate_actions:
    print(action["name"])

Inspect pipeline logs
Inspect failed pipeline stage
Compare with last successful run
Inspect recent code/test changes
Inspect dependency changes
Inspect CI/environment configuration
Search previous incidents/runbooks
Inspect Docker image/cache changes
Inspect server/session state
Escalate to human
